<a href="https://colab.research.google.com/github/Shawaiz-Project/Data-Analyst/blob/main/Executive_Sales_Diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Executive Sales Performance Diagnostic
### Revenue-Quality Diagnostic for an E-Commerce Leadership Team

| | |
|---|---|
| **Role** | Data Analyst / Analytics Engineer |
| **Dataset** | UCI Online Retail (541,909 transactions, 2010-12 → 2011-12) |
| **Objective** | Find the operational drivers behind revenue, cancellations, customer value and repeat purchasing |
| **Deliverables** | Cleaned CSV/Parquet, KPI tables, validation report, Power BI-ready export layer |
| **Stack** | Python · pandas · NumPy · Plotly (Power BI consumes the exports) |

> **Single source of truth:** every number in this notebook is computed from `Online Retail.xlsx`.
> No placeholder KPIs, no fabricated insights. Every filtering decision is logged and reconciled.

**How to run (Google Colab):**
1. Upload `Online Retail.xlsx` when prompted (or let the loader auto-download it from UCI).
2. Run all cells top-to-bottom.
3. Power BI-ready CSVs land in `outputs/csv/`, Parquet in `outputs/parquet/`.


## 1. Project Setup & Configuration
Installs (Colab-safe), imports, display settings, and a reproducible folder layout.


In [1]:
# --- Package setup (Google Colab already ships pandas/numpy; add plotly + pyarrow) ---
# Uncomment the next line if any package is missing in your environment.
!pip install -q plotly pyarrow openpyxl


In [2]:
import os
import sys
import warnings
import zipfile
import urllib.request
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings("ignore")

# ---- Display / formatting -------------------------------------------------
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# ---- Environment detection -------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    pio.renderers.default = "colab"
except ImportError:
    IN_COLAB = False

# ---- Reproducible project layout ------------------------------------------
PROJECT_DIR = Path(os.environ.get(
    "PROJECT_DIR",
    "/content/day-01-executive-sales-diagnostic" if IN_COLAB else Path.cwd() / "day-01-executive-sales-diagnostic",
))

DIR_RAW     = PROJECT_DIR / "data" / "raw"
DIR_PROC    = PROJECT_DIR / "data" / "processed"
DIR_CSV     = PROJECT_DIR / "outputs" / "csv"
DIR_PARQUET = PROJECT_DIR / "outputs" / "parquet"
DIR_REPORTS = PROJECT_DIR / "reports"
for d in (DIR_RAW, DIR_PROC, DIR_CSV, DIR_PARQUET, DIR_REPORTS):
    d.mkdir(parents=True, exist_ok=True)

# ---- Business constants (single place to audit definitions) ----------------
UCI_URL       = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
RAW_XLSX      = DIR_RAW / "Online Retail.xlsx"
REPEAT_MIN_ORDERS   = 2   # repeat customer = 2+ valid sale orders
LOYAL_MIN_ORDERS    = 4   # loyal customer   = 4+ valid sale orders
CURRENCY            = "GBP"

print(f"python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"numpy    : {np.__version__}")
print(f"plotly   : {pio.renderers.default!r} renderer | in_colab={IN_COLAB}")
print(f"project  : {PROJECT_DIR}")
print(f"started  : {datetime.now():%Y-%m-%d %H:%M:%S}")


python   : 3.13.15
pandas   : 2.2.3
numpy    : 2.1.3
plotly   : 'colab' renderer | in_colab=True
project  : /content/day-01-executive-sales-diagnostic
started  : 2026-09-05 15:55:51


## 2. Load Data

The loader tries, in order:
1. **Local file** already in `data/raw/`
2. **Auto-download** from the UCI ML Repository (works in Colab and locally)
3. **Manual upload** prompt (Colab fallback if the network blocks the download)


In [3]:
def load_raw_dataset() -> pd.DataFrame:
    '''Load Online Retail.xlsx, fetching it from UCI or via upload if absent.'''
    if not RAW_XLSX.exists():
        print("Raw file not found locally - attempting UCI download ...")
        try:
            zip_path = DIR_RAW / "online_retail.zip"
            urllib.request.urlretrieve(UCI_URL, zip_path)
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(DIR_RAW)
            zip_path.unlink()
        except Exception as exc:  # network blocked -> manual upload
            print(f"Auto-download failed ({exc}).")
            if IN_COLAB:
                from google.colab import files
                print("Please upload 'Online Retail.xlsx':")
                uploaded = files.upload()
                for name in uploaded:
                    Path(name).replace(RAW_XLSX)
            else:
                raise

    t0 = datetime.now()
    df = pd.read_excel(RAW_XLSX)
    print(f"Loaded {RAW_XLSX.name} in {(datetime.now() - t0).total_seconds():.1f}s")
    return df

raw = load_raw_dataset()


Raw file not found locally - attempting UCI download ...
Loaded Online Retail.xlsx in 54.3s


In [4]:
# ---- First inspection -------------------------------------------------------
print("=" * 72)
print("RAW DATA INSPECTION")
print("=" * 72)
print(f"shape        : {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"memory       : {raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"columns      : {list(raw.columns)}")
print("\ndtypes:")
print(raw.dtypes)
print("\nhead:")
display(raw.head(3))
print("tail:")
display(raw.tail(3))
print("\ndescriptive statistics (numeric):")
display(raw.describe(include=[np.number]).T)
print("\ndescriptive statistics (text):")
# astype(object) makes this portable across pandas 2.x (object dtype) and 3.x (str dtype)
display(raw.astype(object).describe(include=[object]).T)


RAW DATA INSPECTION
shape        : 541,909 rows x 8 columns
memory       : 126.2 MB
columns      : ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

dtypes:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

head:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom


tail:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,"12,680.00",France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,"12,680.00",France



descriptive statistics (numeric):


,count,mean,std,min,25%,50%,75%,max
Quantity,"541,909.00",9.55,218.08,"-80,995.00",1.00,3.00,10.00,"80,995.00"
UnitPrice,"541,909.00",4.61,96.76,"-11,062.06",1.25,2.08,4.13,"38,970.00"
CustomerID,"406,829.00","15,287.69","1,713.60","12,346.00","13,953.00","15,152.00","16,791.00","18,287.00"



descriptive statistics (text):


,count,unique,top,freq
InvoiceNo,541909,25900,573585,1114
StockCode,541909,4070,85123A,2313
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369
Quantity,541909,722,1,148227
InvoiceDate,541909,23260,2011-10-31 14:41:00,1114
UnitPrice,"541,909.00","1,630.00",1.25,"50,496.00"
CustomerID,"406,829.00","4,372.00","17,841.00","7,983.00"
Country,541909,38,United Kingdom,495478


## 3. Data Dictionary

| Field | Business meaning | Type | Analytical usage | Known data-quality issue |
|---|---|---|---|---|
| `InvoiceNo` | 6-digit invoice number; prefix **C** = cancellation | string | order (grain) key | cancellations mixed with sales in one column |
| `StockCode` | product / SKU code | string | product key | non-product codes: `POST`, `DOT`, `M`, `BANK CHARGES`, `PADS`, `CRUK`, `AMAZONFEE`, `C2` ... |
| `Description` | product name (free text) | string | product label | 1,454 missing; inconsistent casing |
| `Quantity` | units per line (negative = return/cancel) | integer | units, revenue | negative & zero values present |
| `InvoiceDate` | invoice timestamp | datetime | time dimension | partial final month (Dec 2011 ends on the 9th) |
| `UnitPrice` | price per unit in GBP | float | revenue, AOV | zero and negative prices (adjustments/bad debt) |
| `CustomerID` | 5-digit customer identifier | integer (nullable) | customer keys, retention | **~24.8% missing** - cannot silently drop |
| `Country` | customer country | string | geography | dominated by UK; some non-country labels (e.g. "Unspecified", "EIRE") |


In [5]:
data_dictionary = pd.DataFrame([
    ("InvoiceNo",   "Invoice number; 'C' prefix flags a cancellation", "string",  "order-grain key",                 "cancellations share the sales column"),
    ("StockCode",   "Product / SKU code",                              "string",  "product key",                     "non-product service codes (POST, DOT, M, BANK CHARGES, PADS, CRUK, AMAZONFEE, C2)"),
    ("Description", "Product name (free text)",                        "string",  "product label",                   "missing values; inconsistent casing"),
    ("Quantity",    "Units sold per line",                             "integer", "units & revenue",                 "negative (returns) and zero quantities"),
    ("InvoiceDate", "Invoice timestamp",                               "datetime","time dimension",                  "final month is partial (Dec 2011 = 9 days)"),
    ("UnitPrice",   "Unit price (GBP)",                                "float",   "revenue & AOV",                   "zero / negative prices (adjustments)"),
    ("CustomerID",  "Customer identifier",                             "Int64",   "customer keys & retention",       "~24.8% missing - keep for transaction analysis"),
    ("Country",     "Customer country",                                "string",  "geography",                       "UK-dominated; few non-standard labels"),
], columns=["field", "business_meaning", "dtype", "analytical_usage", "quality_issue"])

display(data_dictionary)


,field,business_meaning,dtype,analytical_usage,quality_issue
0,InvoiceNo,Invoice number; 'C' prefix flags a cancellation,string,order-grain key,cancellations share the sales column
1,StockCode,Product / SKU code,string,product key,"non-product service codes (POST, DOT, M, BANK ..."
2,Description,Product name (free text),string,product label,missing values; inconsistent casing
3,Quantity,Units sold per line,integer,units & revenue,negative (returns) and zero quantities
4,InvoiceDate,Invoice timestamp,datetime,time dimension,final month is partial (Dec 2011 = 9 days)
5,UnitPrice,Unit price (GBP),float,revenue & AOV,zero / negative prices (adjustments)
6,CustomerID,Customer identifier,Int64,customer keys & retention,~24.8% missing - keep for transaction analysis
7,Country,Customer country,string,geography,UK-dominated; few non-standard labels


## 4. Data Quality Audit

Every anomaly is **counted and logged** before any cleaning decision is taken.
Nothing is removed silently - the audit table below is itself exported as
`12_data_quality_report.csv`.


In [6]:
def audit_data_quality(df: pd.DataFrame) -> pd.DataFrame:
    '''Structured data-quality audit. Returns one row per check.'''
    inv   = df["InvoiceNo"].astype(str)
    stock = df["StockCode"].astype(str)
    qty   = df["Quantity"]
    price = df["UnitPrice"]

    n = len(df)
    checks = [
        ("missing_description",        df["Description"].isna().sum(),        "warning", "label as UNKNOWN PRODUCT"),
        ("missing_customer_id",        df["CustomerID"].isna().sum(),         "warning", "keep rows; exclude only from customer-level analysis"),
        ("duplicate_full_rows",        df.duplicated().sum(),                 "high",    "remove exact duplicates (logging count)"),
        ("duplicate_invoice_product",  df.duplicated(subset=["InvoiceNo", "StockCode", "Quantity", "UnitPrice"]).sum(), "info", "inspect; usually legitimate repeat lines"),
        ("negative_quantity",          (qty < 0).sum(),                       "info",    "cancellations / adjustments - keep, classify"),
        ("zero_quantity",              (qty == 0).sum(),                      "high",    "no commercial value -> classify INVALID"),
        ("negative_unit_price",        (price < 0).sum(),                     "high",    "bad-debt adjustments -> classify ADJUSTMENT"),
        ("zero_unit_price",            (price == 0).sum(),                    "info",    "free / manual lines -> classify INVALID (qty>0) or OTHER"),
        ("cancellation_invoices",      inv.str.startswith("C").sum(),         "info",    "keep; analyse as CANCELLATION"),
        ("non_product_stockcodes",     (~stock.str.match(r"^\d{5}[A-Za-z]?$")).sum(), "info", "flag service/adjustment codes"),
        ("stockcode_whitespace",       (stock != stock.str.strip()).sum(),    "low",     "strip whitespace"),
        ("invalid_dates",              df["InvoiceDate"].isna().sum(),        "high",    "none expected - verify"),
    ]
    rep = pd.DataFrame(checks, columns=["check", "affected_rows", "severity", "treatment"])
    rep["pct_of_rows"] = rep["affected_rows"] / n * 100
    return rep[["check", "affected_rows", "pct_of_rows", "severity", "treatment"]]

quality_report = audit_data_quality(raw)
display(quality_report)

print(f"date range : {raw['InvoiceDate'].min()}  ->  {raw['InvoiceDate'].max()}")
print(f"unique invoices : {raw['InvoiceNo'].nunique():,}")
print(f"unique products : {raw['StockCode'].nunique():,}")
print(f"unique customers: {raw['CustomerID'].nunique():,}")
print(f"countries       : {raw['Country'].nunique():,}")


,check,affected_rows,pct_of_rows,severity,treatment
0,missing_description,1454,0.27,warning,label as UNKNOWN PRODUCT
1,missing_customer_id,135080,24.93,warning,keep rows; exclude only from customer-level an...
2,duplicate_full_rows,5268,0.97,high,remove exact duplicates (logging count)
3,duplicate_invoice_product,5271,0.97,info,inspect; usually legitimate repeat lines
4,negative_quantity,10624,1.96,info,"cancellations / adjustments - keep, classify"
5,zero_quantity,0,0.00,high,no commercial value -> classify INVALID
6,negative_unit_price,2,0.00,high,bad-debt adjustments -> classify ADJUSTMENT
7,zero_unit_price,2515,0.46,info,free / manual lines -> classify INVALID (qty>0...
8,cancellation_invoices,9288,1.71,info,keep; analyse as CANCELLATION
9,non_product_stockcodes,3385,0.62,info,flag service/adjustment codes


date range : 2010-12-01 08:26:00  ->  2011-12-09 12:50:00
unique invoices : 25,900
unique products : 4,070
unique customers: 4,372
countries       : 38


In [7]:
# ---- Inspect the odd codes explicitly (they pollute product KPIs if ignored) ----
stock = raw["StockCode"].astype(str).str.strip()
service_mask = ~stock.str.match(r"^\d{5}[A-Za-z]?$")
service_codes = (raw.loc[service_mask]
                   .assign(StockCode=stock[service_mask])
                   .groupby(["StockCode", raw["Description"].fillna("(no description)")], observed=True)
                   .agg(rows=("Quantity", "size"), units=("Quantity", "sum"))
                   .sort_values("rows", ascending=False)
                   .head(15))
print("Most frequent NON-PRODUCT stock codes (fees, postage, manual adjustments):")
display(service_codes)

print("Example of duplicated full rows (kept visible before removal):")
display(raw[raw.duplicated(keep=False)].sort_values(list(raw.columns)).head(6))


Most frequent NON-PRODUCT stock codes (fees, postage, manual adjustments):


,,rows,units
StockCode,Description,,
POST,POSTAGE,1252,3003
DOT,DOTCOM POSTAGE,709,707
M,Manual,571,3164
15056BL,EDWARDIAN PARASOL BLACK,326,2714
C2,CARRIAGE,143,140
D,Discount,77,-1194
S,SAMPLES,63,-59
15056bl,EDWARDIAN PARASOL BLACK,62,87
BANK CHARGES,Bank Charges,37,-13


Example of duplicated full rows (kept visible before removal):


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,"17,908.00",United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,"17,908.00",United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,"17,908.00",United Kingdom


## 5. Business Rules & Transaction Classification

Every line is classified into exactly one bucket. The rules are **explicit and
order-dependent** - the first rule that matches wins:

| Class | Definition | Rationale |
|---|---|---|
| `CANCELLATION` | `InvoiceNo` starts with **C** | e-commerce convention for this dataset |
| `ADJUSTMENT`  | non-cancel line with `UnitPrice < 0` **or** service code (`BANK CHARGES`, `M`, `C2`...) with `Quantity <= 0` | bad-debt / fee adjustments are not product demand |
| `INVALID`     | non-cancel line with `Quantity == 0`, or (`Quantity > 0` and `UnitPrice == 0`) | zero-value lines carry no revenue signal |
| `SALE`        | everything else: positive quantity, positive price | genuine product sales |

Flags kept on **every** row: `is_cancelled`, `is_valid_quantity`, `is_valid_price`,
`has_customer_id`, `is_service_code`, `is_unknown_customer`, `is_valid_sale`.

> **Design decision:** cancellations and unknown customers are **never deleted** -
> they are classified, so revenue-quality and cancellation analysis stay possible.


In [8]:
NON_PRODUCT_CODES = {
    "POST", "DOT", "M", "BANK CHARGES", "PADS", "CRUK", "AMAZONFEE", "C2", "DCGSSBOY", "DCGSSGIRL", "gift_0001_"
}
NON_PRODUCT_PREFIXES = ("gift_0001",)

def add_business_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # -- normalize types first so flags are deterministic --------------------
    out["InvoiceNo"]  = out["InvoiceNo"].astype(str).str.strip()
    out["StockCode"]  = out["StockCode"].astype(str).str.strip()
    out["Country"]    = out["Country"].astype(str).str.strip()
    out["Quantity"]   = pd.to_numeric(out["Quantity"],  errors="coerce")
    out["UnitPrice"]  = pd.to_numeric(out["UnitPrice"], errors="coerce")
    out["InvoiceDate"] = pd.to_datetime(out["InvoiceDate"], errors="coerce")
    out["CustomerID"] = pd.to_numeric(out["CustomerID"], errors="coerce").astype("Int64")

    # -- atomic flags ----------------------------------------------------------
    out["is_cancelled"]       = out["InvoiceNo"].str.startswith("C")
    out["is_valid_quantity"]  = out["Quantity"] > 0
    out["is_valid_price"]     = out["UnitPrice"] > 0
    out["has_customer_id"]    = out["CustomerID"].notna()
    out["is_service_code"]    = (~out["StockCode"].str.match(r"^\d{5}[A-Za-z]?$")
                                 | out["StockCode"].isin(NON_PRODUCT_CODES)
                                 | out["StockCode"].str.startswith(NON_PRODUCT_PREFIXES))
    out["is_unknown_customer"] = ~out["has_customer_id"]

    # -- ordered classification (first match wins) ------------------------------
    conditions = [
        out["is_cancelled"],
        (out["UnitPrice"] < 0) | (out["is_service_code"] & (out["Quantity"] <= 0)),
        (out["Quantity"] == 0) | (~out["is_valid_price"]),
    ]
    out["transaction_class"] = np.select(conditions, ["CANCELLATION", "ADJUSTMENT", "INVALID"], default="SALE")
    out["is_valid_sale"] = out["transaction_class"] == "SALE"
    return out

flagged = add_business_flags(raw)

_v = flagged.assign(_signed=flagged["Quantity"] * flagged["UnitPrice"])
class_summary = (_v.groupby("transaction_class", as_index=True)
                   .agg(rows=("InvoiceNo", "size"),
                        units=("Quantity", "sum"),
                        signed_value_gbp=("_signed", "sum")))
class_summary["pct_rows"] = class_summary["rows"] / len(flagged) * 100
display(class_summary)
assert class_summary["rows"].sum() == len(flagged), "classification must partition ALL rows"


,rows,units,signed_value_gbp,pct_rows
transaction_class,,,,
ADJUSTMENT,16,-3833,"-22,124.12",0.00
CANCELLATION,9288,-277574,"-896,812.49",1.71
INVALID,2501,-130519,0.00,0.46
SALE,530104,5588376,"10,666,684.54",97.82


## 6. Cleaning

Cleaning = **de-duplicate + standardize**, never silent deletion.

Rules applied (each logged):
1. Drop **exact duplicate rows** (same invoice, product, qty, price, date, customer) - count logged.
2. Fill missing `Description` with `UNKNOWN PRODUCT` (labels only, no metric impact).
3. Keep missing `CustomerID` rows for transaction-level analysis; they are excluded **only** from customer-level tables.
4. `ADJUSTMENT` and `INVALID` rows are excluded from the *analytical* fact table but remain in `flagged` for audit.
5. Cancellations stay in the analytical fact table (flagged, not deleted).

The result is `fact` = the analytical transaction table, with a full audit trail.


In [9]:
cleaning_log = []

def log_step(log, step, before, after, note=""):
    log.append({"step": step, "rows_before": before, "rows_after": after,
                "rows_removed": before - after, "note": note})
    return after

# ---- step 1: exact duplicates -------------------------------------------------
n0 = len(flagged)
clean = flagged.drop_duplicates().copy()
log_step(cleaning_log, "remove exact duplicate rows", n0, len(clean),
         "identical invoice/product/qty/price/date/customer lines")

# ---- step 2: standardize labels --------------------------------------------------
clean["Description"] = (clean["Description"].fillna("UNKNOWN PRODUCT")
                                            .astype(str).str.strip().str.upper())
log_step(cleaning_log, "standardize Description (fillna + upper)", len(clean), len(clean),
         "no rows removed")

# ---- step 3: build the analytical fact table ---------------------------------------
n0 = len(clean)
fact = clean[clean["transaction_class"].isin(["SALE", "CANCELLATION"])].copy()
log_step(cleaning_log, "keep SALE + CANCELLATION in analytical fact", n0, len(fact),
         "ADJUSTMENT/INVALID stay available in `flagged` for audit")

cleaning_report = pd.DataFrame(cleaning_log)
display(cleaning_report)
print(f"\nAnalytical fact table: {len(fact):,} rows "
      f"({fact['is_valid_sale'].sum():,} sales, {(~fact['is_valid_sale']).sum():,} cancellations)")


,step,rows_before,rows_after,rows_removed,note
0,remove exact duplicate rows,541909,536641,5268,identical invoice/product/qty/price/date/custo...
1,standardize Description (fillna + upper),536641,536641,0,no rows removed
2,keep SALE + CANCELLATION in analytical fact,536641,534129,2512,ADJUSTMENT/INVALID stay available in `flagged`...



Analytical fact table: 534,129 rows (524,878 sales, 9,251 cancellations)


## 7. Revenue Engineering

Signed truth first, reporting views second - **gross and net are never mixed**.

| Column | Formula | Meaning |
|---|---|---|
| `signed_revenue` | `Quantity × UnitPrice` | raw economic value (cancellations negative) |
| `gross_revenue`  | `signed_revenue` where `SALE` else 0 | demand generated |
| `cancellation_value` | `|signed_revenue|` where `CANCELLATION` else 0 | revenue lost to cancellations (positive for reporting) |
| `net_revenue` | `gross_revenue − cancellation_value` | actual booked revenue |
| `valid_sales_revenue` | `gross_revenue` where customer identified | revenue usable for customer-level analysis |

`signed_revenue` is always preserved, so totals reconcile exactly to the source.


In [10]:
def engineer_revenue(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["signed_revenue"]     = out["Quantity"] * out["UnitPrice"]
    out["gross_revenue"]      = np.where(out["transaction_class"] == "SALE", out["signed_revenue"], 0.0)
    out["cancellation_value"] = np.where(out["transaction_class"] == "CANCELLATION",
                                         out["signed_revenue"].abs(), 0.0)
    out["net_revenue"]        = out["gross_revenue"] - out["cancellation_value"]
    out["valid_sales_revenue"] = np.where(out["is_valid_sale"] & out["has_customer_id"],
                                          out["gross_revenue"], 0.0)
    return out

fact = engineer_revenue(fact)

rev_check = pd.DataFrame({
    "measure": ["gross_revenue", "cancellation_value", "net_revenue", "signed_revenue"],
    "value_gbp": [fact["gross_revenue"].sum(), fact["cancellation_value"].sum(),
                  fact["net_revenue"].sum(),   fact["signed_revenue"].sum()],
})
display(rev_check)
print("sanity: net == signed ?",
      np.isclose(fact["net_revenue"].sum(), fact["signed_revenue"].sum()))


,measure,value_gbp
0,gross_revenue,"10,642,110.80"
1,cancellation_value,"893,979.73"
2,net_revenue,"9,748,131.07"
3,signed_revenue,"9,748,131.07"


sanity: net == signed ? True


## 8. Order-Level Dataset (`orders`)

Grain: **one row per invoice**. AOV is computed at *this* grain - never from
transaction row counts (a classic failure mode).


In [11]:
def build_orders(f: pd.DataFrame) -> pd.DataFrame:
    g = f.groupby("InvoiceNo", as_index=False).agg(
        InvoiceDate     = ("InvoiceDate", "min"),
        CustomerID      = ("CustomerID", "first"),
        Country         = ("Country", "first"),
        order_revenue   = ("net_revenue", "sum"),
        gross_revenue   = ("gross_revenue", "sum"),
        cancellation_value = ("cancellation_value", "sum"),
        order_units     = ("Quantity", "sum"),
        order_line_count= ("StockCode", "size"),
        is_cancelled    = ("is_cancelled", "max"),
        has_customer_id = ("has_customer_id", "max"),
    )
    g["is_sale_order"] = ~g["is_cancelled"]
    return g

orders = build_orders(fact)

sale_orders = orders[orders["is_sale_order"]]
AOV = sale_orders["order_revenue"].sum() / sale_orders["InvoiceNo"].nunique()

print(f"orders            : {len(orders):,} "
      f"({orders['is_sale_order'].sum():,} sales, {(~orders['is_sale_order']).sum():,} cancellations)")
print(f"AOV (valid sales) : £{AOV:,.2f}  "
      f"= £{sale_orders['order_revenue'].sum():,.0f} / {sale_orders['InvoiceNo'].nunique():,} orders")
display(orders.head(3))


orders            : 23,796 (19,960 sales, 3,836 cancellations)
AOV (valid sales) : £533.17  = £10,642,111 / 19,960 orders


,InvoiceNo,InvoiceDate,CustomerID,Country,order_revenue,gross_revenue,cancellation_value,order_units,order_line_count,is_cancelled,has_customer_id,is_sale_order
0,536365,2010-12-01 08:26:00,17850,United Kingdom,139.12,139.12,0.00,40,7,False,True,True
1,536366,2010-12-01 08:28:00,17850,United Kingdom,22.20,22.20,0.00,12,2,False,True,True
2,536367,2010-12-01 08:34:00,13047,United Kingdom,278.73,278.73,0.00,83,12,False,True,True


## 9. Customer-Level Dataset (`customers`)

Grain: **one row per identified customer** (`has_customer_id == True` only).
Unknown customers stay in the fact table for revenue, but retention metrics
require an identity, so they are excluded **here only** — explicitly, not silently.

**Repeat customer definition:** a customer with **2+ valid sale orders**
(`is_sale_order`) over their observed lifetime.


In [12]:
def build_customers(orders_df: pd.DataFrame, facts: pd.DataFrame) -> pd.DataFrame:
    # customer grain = identified customers only
    so = orders_df[orders_df["has_customer_id"] & orders_df["is_sale_order"]]
    co = orders_df[orders_df["has_customer_id"] & ~orders_df["is_sale_order"]]

    cust = so.groupby("CustomerID", as_index=False).agg(
        Country             = ("Country", "first"),
        first_purchase_date = ("InvoiceDate", "min"),
        last_purchase_date  = ("InvoiceDate", "max"),
        order_count         = ("InvoiceNo", "nunique"),
        total_units         = ("order_units", "sum"),
        gross_revenue       = ("order_revenue", "sum"),
    )
    canc = (co.groupby("CustomerID")["cancellation_value"].sum()
              .rename("cancellation_value").reset_index())
    cust = cust.merge(canc, on="CustomerID", how="left").fillna({"cancellation_value": 0.0})

    cust["net_revenue"] = cust["gross_revenue"] - cust["cancellation_value"]
    cust["average_order_value"] = np.where(cust["order_count"] > 0,
                                           cust["gross_revenue"] / cust["order_count"], 0.0)
    cust["customer_lifetime_days"] = (cust["last_purchase_date"]
                                      - cust["first_purchase_date"]).dt.days
    cust["purchase_frequency"] = np.where(
        cust["customer_lifetime_days"] > 0,
        cust["order_count"] / (cust["customer_lifetime_days"] / 30.4375),  # orders per month
        cust["order_count"].astype(float),                                  # single-day customers
    )
    cust["is_repeat_customer"] = cust["order_count"] >= REPEAT_MIN_ORDERS

    # ---- segmentation: transparent business rules -----------------------------
    cust["customer_segment"] = np.select(
        [cust["order_count"] >= LOYAL_MIN_ORDERS, cust["order_count"] >= REPEAT_MIN_ORDERS],
        ["Loyal", "Returning"],
        default="One-Time",
    )

    # ---- RFM-style scores (diagnostic, not ML) ---------------------------------
    snapshot = facts["InvoiceDate"].max() + pd.Timedelta(days=1)
    cust["recency_days"] = (snapshot - cust["last_purchase_date"]).dt.days
    def _qcut(s, labels):  # rank(method='first') breaks ties so qcut never collapses
        return pd.qcut(s.rank(method="first"), 5, labels=labels).astype(int)
    cust["r_score"] = _qcut(-cust["recency_days"], [5, 4, 3, 2, 1])  # recent = high score
    cust["f_score"] = _qcut(cust["order_count"],  [1, 2, 3, 4, 5])
    cust["m_score"] = _qcut(cust["net_revenue"], [1, 2, 3, 4, 5])
    cust["rfm_score"] = cust["r_score"] + cust["f_score"] + cust["m_score"]
    return cust

customers = build_customers(orders, fact)

repeat_rate = customers["is_repeat_customer"].mean() * 100
print(f"identified customers : {len(customers):,}")
print(f"repeat customers     : {customers['is_repeat_customer'].sum():,} "
      f"({repeat_rate:.1f}% of identified customers)")
display(customers["customer_segment"].value_counts().rename_axis("segment").to_frame("customers"))
display(customers.head(3))


identified customers : 4,338
repeat customers     : 2,845 (65.6% of identified customers)


,customers
segment,
Loyal,1502
One-Time,1493
Returning,1343


,CustomerID,Country,first_purchase_date,last_purchase_date,order_count,total_units,gross_revenue,cancellation_value,net_revenue,average_order_value,customer_lifetime_days,purchase_frequency,is_repeat_customer,customer_segment,recency_days,r_score,f_score,m_score,rfm_score
0,12346,United Kingdom,2011-01-18 10:01:00,2011-01-18 10:01:00,1,74215,"77,183.60","77,183.60",0.00,"77,183.60",0,1.00,False,One-Time,326,5,1,1,7
1,12347,Iceland,2010-12-07 14:57:00,2011-12-07 15:52:00,7,2458,"4,310.00",0.00,"4,310.00",615.71,365,0.58,True,Loyal,2,1,5,5,11
2,12348,Finland,2010-12-16 19:09:00,2011-09-25 13:13:00,4,2341,"1,797.24",0.00,"1,797.24",449.31,282,0.43,True,Loyal,75,4,4,4,12


## 10. Segmentation Rules (documented)

| Segment | Rule | Business meaning |
|---|---|---|
| **One-Time** | exactly 1 valid sale order | transacted once, never returned |
| **Returning** | 2–3 valid sale orders | demonstrated repeat intent |
| **Loyal** | 4+ valid sale orders | habitual buyers; core revenue base |

RFM scores (1–5 quintiles on Recency, Frequency, Monetary) are included as a
**diagnostic overlay** only — no ML clustering, per project scope.


## 11. Country Analysis

Grain: one row per country. Cancellations and repeat behaviour are expressed as
rates so countries of very different sizes stay comparable.


In [13]:
def build_country_kpis(orders_df: pd.DataFrame, customers_df: pd.DataFrame,
                       facts: pd.DataFrame) -> pd.DataFrame:
    so = orders_df[orders_df["is_sale_order"]]
    co = orders_df[~orders_df["is_sale_order"]]

    ctry = so.groupby("Country", as_index=False).agg(
        orders        = ("InvoiceNo", "nunique"),
        customers     = ("CustomerID", "nunique"),
        units         = ("order_units", "sum"),
        gross_revenue = ("order_revenue", "sum"),
    )
    canc = (co.groupby("Country")
              .agg(cancel_orders=("InvoiceNo", "nunique"),
                   cancellation_value=("cancellation_value", "sum")).reset_index())
    ctry = ctry.merge(canc, on="Country", how="left").fillna(
        {"cancel_orders": 0, "cancellation_value": 0.0})

    # repeat rate per country from the customer table (identified customers only)
    rep = (customers_df.groupby("Country")
                       .agg(identified_customers=("CustomerID", "size"),
                            repeat_customers=("is_repeat_customer", "sum")).reset_index())
    ctry = ctry.merge(rep, on="Country", how="left").fillna(
        {"identified_customers": 0, "repeat_customers": 0})

    ctry["net_revenue"]        = ctry["gross_revenue"] - ctry["cancellation_value"]
    ctry["aov"]                = ctry["gross_revenue"] / ctry["orders"]
    ctry["revenue_per_customer"] = np.where(ctry["identified_customers"] > 0,
                                            ctry["net_revenue"] / ctry["identified_customers"], np.nan)
    ctry["repeat_customer_rate"] = np.where(ctry["identified_customers"] > 0,
                                            ctry["repeat_customers"] / ctry["identified_customers"] * 100, np.nan)
    ctry["cancellation_rate"]  = np.where(ctry["orders"] + ctry["cancel_orders"] > 0,
                                          ctry["cancel_orders"] / (ctry["orders"] + ctry["cancel_orders"]) * 100, 0.0)
    return ctry.sort_values("net_revenue", ascending=False).reset_index(drop=True)

country_kpis = build_country_kpis(orders, customers, fact)

print("Top 10 countries by NET REVENUE:")
display(country_kpis.head(10)[["Country", "net_revenue", "orders", "customers", "aov",
                               "repeat_customer_rate", "cancellation_rate"]])
print("Top 5 countries by CANCELLATION RATE (min 100 orders):")
display(country_kpis[country_kpis["orders"] >= 100]
        .nlargest(5, "cancellation_rate")
        [["Country", "cancellation_rate", "cancel_orders", "cancellation_value"]])
print("Top 5 countries by REPEAT RATE (min 50 identified customers):")
display(country_kpis[country_kpis["identified_customers"] >= 50]
        .nlargest(5, "repeat_customer_rate")
        [["Country", "repeat_customer_rate", "repeat_customers", "identified_customers"]])


Top 10 countries by NET REVENUE:


,Country,net_revenue,orders,customers,aov,repeat_customer_rate,cancellation_rate
0,United Kingdom,"8,189,252.30",18019,3920,499.57,65.56,15.76
1,Netherlands,"284,661.54",94,9,"3,036.66",55.56,6.00
2,EIRE,"262,993.38",288,3,983.13,100.00,20.00
3,Germany,"221,509.47",457,94,500.39,72.34,24.21
4,France,"197,317.11",392,87,534.76,67.82,14.97
5,Australia,"137,009.77",57,9,"2,429.01",100.00,17.39
6,Switzerland,"56,363.05",54,21,"1,056.81",75.00,27.03
7,Spain,"54,756.03",90,30,683.98,64.29,14.29
8,Belgium,"40,910.96",98,25,420.37,75.00,17.65
9,Sweden,"36,585.41",36,8,"1,065.77",50.00,21.74


Top 5 countries by CANCELLATION RATE (min 100 orders):


,Country,cancellation_rate,cancel_orders,cancellation_value
3,Germany,24.21,146.00,"7,168.93"
2,EIRE,20.00,72.00,"20,147.14"
0,United Kingdom,15.76,"3,372.00","812,491.79"
4,France,14.97,69.00,"12,308.26"


Top 5 countries by REPEAT RATE (min 50 identified customers):


,Country,repeat_customer_rate,repeat_customers,identified_customers
3,Germany,72.34,68.00,94
4,France,67.82,59.00,87
0,United Kingdom,65.56,"2,570.00",3920


## 12. Product Analysis

Grain: one row per `StockCode`. Service/fee codes are excluded from product KPIs
(`is_service_code == False`) so postage and bank charges never masquerade as
products.


In [14]:
def build_product_kpis(facts: pd.DataFrame) -> pd.DataFrame:
    f = facts[~facts["is_service_code"]].copy()
    f["sale_units"] = np.where(f["is_valid_sale"], f["Quantity"], 0)   # avoid fragile index lambdas
    prod = f.groupby(["StockCode", "Description"], as_index=False).agg(
        orders        = ("InvoiceNo", "nunique"),
        customers     = ("CustomerID", "nunique"),
        units         = ("sale_units", "sum"),
        gross_revenue = ("gross_revenue", "sum"),
        cancellation_value = ("cancellation_value", "sum"),
        avg_unit_price= ("UnitPrice", "mean"),
    )
    # cancellation units & counts for rate
    canc = (f[~f["is_valid_sale"]].groupby("StockCode")
            .agg(cancel_lines=("InvoiceNo", "size")).reset_index())
    tot  = (f.groupby("StockCode").agg(total_lines=("InvoiceNo", "size")).reset_index())
    prod = prod.merge(canc, on="StockCode", how="left").merge(tot, on="StockCode")
    prod["cancel_lines"] = prod["cancel_lines"].fillna(0)
    prod["net_revenue"]  = prod["gross_revenue"] - prod["cancellation_value"]
    prod["cancellation_rate"] = np.where(prod["total_lines"] > 0,
                                         prod["cancel_lines"] / prod["total_lines"] * 100, 0.0)
    return prod.sort_values("net_revenue", ascending=False).reset_index(drop=True)

product_kpis = build_product_kpis(fact)

print("TOP 10 PRODUCTS by NET REVENUE:")
display(product_kpis.head(10)[["StockCode", "Description", "net_revenue", "units", "orders"]])
print("TOP 10 PRODUCTS by UNITS:")
display(product_kpis.nlargest(10, "units")[["StockCode", "Description", "units", "net_revenue"]])
print("HIGHEST CANCELLATION-VALUE PRODUCTS (min £1,000 cancelled):")
display(product_kpis[product_kpis["cancellation_value"] >= 1000]
        .nlargest(10, "cancellation_value")
        [["StockCode", "Description", "cancellation_value", "cancellation_rate"]])


TOP 10 PRODUCTS by NET REVENUE:


,StockCode,Description,net_revenue,units,orders
0,22423,REGENCY CAKESTAND 3 TIER,"164,459.49",13851,2168
1,47566,PARTY BUNTING,"98,243.88",18283,1705
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"97,659.94",37580,2231
3,85099B,JUMBO BAG RED RETROSPOT,"92,175.79",48371,2132
4,23084,RABBIT NIGHT LIGHT,"66,661.63",30739,1009
5,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"63,715.24",19329,1170
6,84879,ASSORTED COLOUR BIRD ORNAMENT,"58,792.42",36362,1467
7,79321,CHILLI LIGHTS,"53,746.66",10302,667
8,23298,SPOTTY BUNTING,"42,030.67",8320,1152
9,22386,JUMBO BAG PINK POLKADOT,"41,584.43",21448,1231


TOP 10 PRODUCTS by UNITS:


,StockCode,Description,units,net_revenue
4128,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,0.00
581,23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,"4,221.28"
137,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54951,"13,560.09"
3,85099B,JUMBO BAG RED RETROSPOT,48371,"92,175.79"
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,"97,659.94"
17,22197,POPCORN HOLDER,36749,"33,959.26"
77,21212,PACK OF 72 RETROSPOT CAKE CASES,36396,"21,047.07"
6,84879,ASSORTED COLOUR BIRD ORNAMENT,36362,"58,792.42"
4,23084,RABBIT NIGHT LIGHT,30739,"66,661.63"
106,22492,MINI PAINT SET VINTAGE,26633,"16,810.42"


HIGHEST CANCELLATION-VALUE PRODUCTS (min £1,000 cancelled):


,StockCode,Description,cancellation_value,cancellation_rate
4128,23843,"PAPER CRAFT , LITTLE BIRDIE","168,469.60",50.00
581,23166,MEDIUM CERAMIC TOP STORAGE JAR,"77,479.64",3.85
0,22423,REGENCY CAKESTAND 3 TIER,"9,697.05",8.23
2,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"6,624.30",1.83
175,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,"6,591.42",1.13
1567,23113,PANTRY CHOPPING BOARD,"4,803.06",9.52
83,48185,DOORMAT FAIRY CAKE,"4,554.90",0.81
68,21175,GIN + TONIC DIET METAL SIGN,"3,775.33",0.85
305,47566B,TEA TIME PARTY BUNTING,"3,692.95",1.88
270,22273,FELTCRAFT DOLL MOLLY,"3,512.65",1.77


## 13. Time Series & Date Dimension

`dim_date` is a Power BI-friendly calendar table covering every day in the
dataset range. `monthly_kpis` aggregates the business at `YearMonth` grain.

**Monthly repeat-rate definition:** share of the month's *identified active
customers* whose **first-ever purchase was before that month** (i.e. returning
buyers, not new acquisitions). This is documented because "repeat rate by
month" is otherwise ambiguous.


In [15]:
def build_date_dim(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    d = pd.DataFrame({"date": pd.date_range(start.normalize(), end.normalize(), freq="D")})
    d["date_key"]    = d["date"].dt.strftime("%Y%m%d").astype(int)  # surrogate key for Power BI joins
    d["year"]        = d["date"].dt.year
    d["month"]       = d["date"].dt.month
    d["month_name"]  = d["date"].dt.month_name()
    d["year_month"]  = d["date"].dt.strftime("%Y-%m")
    d["quarter"]     = "Q" + d["date"].dt.quarter.astype(str)
    d["year_quarter"] = d["date"].dt.year.astype(str) + "-Q" + d["date"].dt.quarter.astype(str)
    d["week"]        = d["date"].dt.isocalendar().week.astype(int)
    d["day"]         = d["date"].dt.day
    d["day_of_week"] = d["date"].dt.day_name()
    d["is_weekend"]  = d["date"].dt.dayofweek >= 5
    return d

dim_date = build_date_dim(fact["InvoiceDate"].min(), fact["InvoiceDate"].max())
print(f"date dimension: {len(dim_date):,} days "
      f"({dim_date['date'].min():%Y-%m-%d} -> {dim_date['date'].max():%Y-%m-%d})")
display(dim_date.head(3))


date dimension: 374 days (2010-12-01 -> 2011-12-09)


,date,date_key,year,month,month_name,year_month,quarter,year_quarter,week,day,day_of_week,is_weekend
0,2010-12-01,20101201,2010,12,December,2010-12,Q4,2010-Q4,48,1,Wednesday,False
1,2010-12-02,20101202,2010,12,December,2010-12,Q4,2010-Q4,48,2,Thursday,False
2,2010-12-03,20101203,2010,12,December,2010-12,Q4,2010-Q4,48,3,Friday,False


In [16]:
def build_monthly_kpis(orders_df: pd.DataFrame, customers_df: pd.DataFrame) -> pd.DataFrame:
    o = orders_df.copy()
    o["year_month"] = o["InvoiceDate"].dt.strftime("%Y-%m")
    o["month_start"] = o["InvoiceDate"].values.astype("datetime64[M]")

    so, co = o[o["is_sale_order"]], o[~o["is_sale_order"]]

    m = so.groupby("year_month", as_index=False).agg(
        orders        = ("InvoiceNo", "nunique"),
        customers     = ("CustomerID", "nunique"),
        units         = ("order_units", "sum"),
        gross_revenue = ("order_revenue", "sum"),
    )
    canc = (co.groupby("year_month")
              .agg(cancel_orders=("InvoiceNo", "nunique"),
                   cancellation_value=("cancellation_value", "sum")).reset_index())
    m = m.merge(canc, on="year_month", how="left").fillna(
        {"cancel_orders": 0, "cancellation_value": 0.0})

    # ---- repeat-active customers per month (first purchase strictly before month) ----
    # vectorized: a customer is 'returning' in a month if their FIRST-ever purchase
    # predates that month; computed on distinct (customer, month) pairs, not line lambdas
    first = customers_df.set_index("CustomerID")["first_purchase_date"]
    cm = (so[so["has_customer_id"]]
          .groupby(["year_month", "month_start", "CustomerID"], as_index=False)
          .size())
    cm["first_purchase"] = cm["CustomerID"].map(first)
    cm = cm.dropna(subset=["first_purchase"])
    cm["is_returning_this_month"] = cm["first_purchase"] < cm["month_start"]
    rep = (cm.groupby("year_month", as_index=False)
             .agg(active_identified=("CustomerID", "nunique"),
                  returning_active=("is_returning_this_month", "sum")))
    m = m.merge(rep, on="year_month", how="left").fillna(
        {"active_identified": 0, "returning_active": 0})

    m["net_revenue"]   = m["gross_revenue"] - m["cancellation_value"]
    m["aov"]           = m["gross_revenue"] / m["orders"]
    m["revenue_per_customer"] = np.where(m["active_identified"] > 0,
                                         m["net_revenue"] / m["active_identified"], np.nan)
    m["repeat_customer_rate"] = np.where(m["active_identified"] > 0,
                                         m["returning_active"] / m["active_identified"] * 100, np.nan)
    m["cancellation_rate"]    = np.where(m["orders"] + m["cancel_orders"] > 0,
                                         m["cancel_orders"] / (m["orders"] + m["cancel_orders"]) * 100, 0.0)
    m["orders_per_customer"]  = np.where(m["active_identified"] > 0,
                                         m["orders"] / m["active_identified"], np.nan)
    m = m.sort_values("year_month").reset_index(drop=True)
    m["mom_revenue_growth_pct"] = m["net_revenue"].pct_change() * 100
    return m

monthly_kpis = build_monthly_kpis(orders, customers)
display(monthly_kpis[["year_month", "net_revenue", "orders", "customers", "aov",
                      "repeat_customer_rate", "cancellation_rate", "mom_revenue_growth_pct"]])

print("NOTE: the final month (2011-12) only covers 9 days - interpret its dip accordingly.")


,year_month,net_revenue,orders,customers,aov,repeat_customer_rate,cancellation_rate,mom_revenue_growth_pct
0,2010-12,"746,723.61",1559,885,526.91,0.00,17.29,NaN
1,2011-01,"558,448.56",1086,741,635.19,43.72,19.32,-25.21
2,2011-02,"497,026.41",1100,758,475.04,49.87,16.60,-11.00
3,2011-03,"682,013.98",1454,974,492.58,53.59,17.95,37.22
4,2011-04,"492,367.84",1246,856,430.95,64.95,16.15,-27.81
5,2011-05,"722,094.10",1681,1056,457.64,73.11,15.74,46.66
6,2011-06,"689,977.23",1533,991,496.12,75.58,17.67,-4.45
7,2011-07,"680,156.99",1475,949,486.83,80.19,15.47,-1.42
8,2011-08,"703,510.58",1361,935,556.83,81.93,16.96,3.43
9,2011-09,"1,017,596.68",1837,1266,575.09,76.38,15.35,44.65


NOTE: the final month (2011-12) only covers 9 days - interpret its dip accordingly.


## 14. Revenue Decomposition (business-driver analysis)

Identity used: **Revenue = Active Customers × Orders per Customer × AOV**

Computed month-over-month on **full months only** (Dec 2011 is partial and is
excluded from the decomposition to avoid a false crash signal). Language is
deliberately associative ("associated with", "consistent with") — this is
descriptive evidence, not causal proof.


In [17]:
# Exclude partial EDGE months by data coverage, not by blind positional slicing.
# A month is 'full' if its span covers >= 25 of its calendar days (Dec 2011 = 9 days -> partial).
def _is_full_month(ym):
    d = fact.loc[fact["InvoiceDate"].dt.strftime("%Y-%m") == ym, "InvoiceDate"].dt.day
    return (d.max() - d.min() + 1) >= 25 if len(d) else False

_full_mask = monthly_kpis["year_month"].map(_is_full_month)
FULL_MONTHS = monthly_kpis[_full_mask].copy()
_dropped = monthly_kpis.loc[~_full_mask, "year_month"].tolist()
print(f"partial months excluded from decomposition: {_dropped}")

first_m, last_m = FULL_MONTHS.iloc[0], FULL_MONTHS.iloc[-1]
decomp = pd.DataFrame({
    "driver": ["net_revenue", "active_customers", "orders_per_customer", "aov"],
    f"{first_m['year_month']}": [first_m["net_revenue"], first_m["active_identified"],
                                 first_m["orders_per_customer"], first_m["aov"]],
    f"{last_m['year_month']}":  [last_m["net_revenue"], last_m["active_identified"],
                                 last_m["orders_per_customer"], last_m["aov"]],
})
decomp["change_pct"] = (decomp[f"{last_m['year_month']}"] / decomp[f"{first_m['year_month']}"] - 1) * 100
display(decomp)

rev_growth  = decomp.loc[decomp["driver"] == "net_revenue", "change_pct"].iloc[0]
cust_growth = decomp.loc[decomp["driver"] == "active_customers", "change_pct"].iloc[0]
freq_growth = decomp.loc[decomp["driver"] == "orders_per_customer", "change_pct"].iloc[0]
aov_growth  = decomp.loc[decomp["driver"] == "aov", "change_pct"].iloc[0]

print(f"Between {first_m['year_month']} and {last_m['year_month']} (full months only):")
print(f"  net revenue          : {rev_growth:+.1f}%")
print(f"  active customers     : {cust_growth:+.1f}%")
print(f"  orders per customer  : {freq_growth:+.1f}%")
print(f"  AOV                  : {aov_growth:+.1f}%")
print("\nReading: revenue change is the product of these drivers; whichever moved most")
print("is the driver growth is MOST ASSOCIATED WITH (association, not causation).")


partial months excluded from decomposition: ['2010-12', '2011-12']


,driver,2011-01,2011-11,change_pct
0,net_revenue,"558,448.56","1,456,145.80",160.75
1,active_customers,741.00,"1,664.00",124.56
2,orders_per_customer,1.47,1.66,13.54
3,aov,635.19,543.11,-14.50


Between 2011-01 and 2011-11 (full months only):
  net revenue          : +160.7%
  active customers     : +124.6%
  orders per customer  : +13.5%
  AOV                  : -14.5%

Reading: revenue change is the product of these drivers; whichever moved most
is the driver growth is MOST ASSOCIATED WITH (association, not causation).


## 15. Cancellation Analysis

Where is revenue leaking? Cancellations sliced by month, country, product, customer.
Cancellation **rate** = cancellation invoices / (sale + cancellation invoices).


In [18]:
canc_fact = fact[fact["transaction_class"] == "CANCELLATION"]

cancel_by_month = (monthly_kpis[["year_month", "cancel_orders", "cancellation_value",
                                 "cancellation_rate"]]
                   .rename(columns={"cancel_orders": "cancellation_count"}))
cancel_by_country = (country_kpis[["Country", "cancel_orders", "cancellation_value",
                                   "cancellation_rate", "orders"]]
                     .rename(columns={"cancel_orders": "cancellation_count"})
                     .sort_values("cancellation_value", ascending=False))
cancel_by_product = (product_kpis[["StockCode", "Description", "cancel_lines",
                                   "cancellation_value", "cancellation_rate"]]
                     .rename(columns={"cancel_lines": "cancellation_count"})
                     .query("cancellation_value > 0")
                     .sort_values("cancellation_value", ascending=False))
cancel_by_customer = (customers[customers["cancellation_value"] > 0]
                      [["CustomerID", "Country", "cancellation_value", "net_revenue"]]
                      .sort_values("cancellation_value", ascending=False))

total_canc = fact["cancellation_value"].sum()
total_gross = fact["gross_revenue"].sum()
print(f"total cancellation value : £{total_canc:,.0f}  "
      f"({total_canc / total_gross * 100:.1f}% of gross sales)")
print(f"cancellation invoices    : {canc_fact['InvoiceNo'].nunique():,}")
print("\nworst month by cancellation rate:")
display(cancel_by_month.nlargest(3, "cancellation_rate"))
print("worst countries by cancellation value:")
display(cancel_by_country.head(5))
print("customers with the largest cancelled value:")
display(cancel_by_customer.head(5))


total cancellation value : £893,980  (8.4% of gross sales)
cancellation invoices    : 3,836

worst month by cancellation rate:


,year_month,cancellation_count,cancellation_value,cancellation_rate
1,2011-01,260,"131,363.05",19.32
3,2011-03,318,"34,201.28",17.95
6,2011-06,329,"70,569.78",17.67


worst countries by cancellation value:


,Country,cancellation_count,cancellation_value,cancellation_rate,orders
0,United Kingdom,"3,372.00","812,491.79",15.76,18019
2,EIRE,72.00,"20,147.14",20.00,288
4,France,69.00,"12,308.26",14.97,392
20,Singapore,3.00,"12,158.90",30.00,7
3,Germany,146.00,"7,168.93",24.21,457


customers with the largest cancelled value:


,CustomerID,Country,cancellation_value,net_revenue
3008,16446,United Kingdom,"168,469.60",2.90
0,12346,United Kingdom,"77,183.60",0.00
2011,15098,United Kingdom,"39,267.00",649.50
2702,16029,United Kingdom,"27,682.15","53,168.69"
2502,15749,United Kingdom,"22,998.40","21,535.90"


## 16. Customer Revenue Concentration

How dependent is the business on a small set of buyers? We compute the revenue
share of the top 10 customers and of the top 10% / 20% of customers.


In [19]:
conc = customers.sort_values("net_revenue", ascending=False).reset_index(drop=True)
total_net_cust = conc["net_revenue"].sum()

top10_customers  = conc.head(10)[["CustomerID", "Country", "order_count",
                                  "net_revenue", "customer_segment"]]
top10_share  = conc.head(10)["net_revenue"].sum() / total_net_cust * 100
n10 = int(np.ceil(len(conc) * 0.10)); n20 = int(np.ceil(len(conc) * 0.20))
top10pct_share = conc.head(n10)["net_revenue"].sum() / total_net_cust * 100
top20pct_share = conc.head(n20)["net_revenue"].sum() / total_net_cust * 100

concentration_summary = pd.DataFrame({
    "cohort": ["Top 10 customers", "Top 10% of customers", "Top 20% of customers"],
    "customers": [10, n10, n20],
    "net_revenue_share_pct": [top10_share, top10pct_share, top20pct_share],
})
display(top10_customers)
display(concentration_summary)
print(f"identified-customer net revenue: £{total_net_cust:,.0f}")
print(f"company net revenue            : £{fact['net_revenue'].sum():,.0f} "
      f"(difference = unidentified-customer revenue, expected)")


,CustomerID,Country,order_count,net_revenue,customer_segment
0,14646,Netherlands,73,"279,489.02",Loyal
1,18102,United Kingdom,60,"256,438.49",Loyal
2,17450,United Kingdom,46,"187,322.17",Loyal
3,14911,EIRE,201,"132,458.73",Loyal
4,12415,Australia,21,"123,725.45",Loyal
5,14156,EIRE,55,"113,214.59",Loyal
6,17511,United Kingdom,31,"88,125.38",Loyal
7,16684,United Kingdom,28,"65,892.08",Loyal
8,13694,United Kingdom,50,"62,690.54",Loyal
9,15311,United Kingdom,91,"59,284.19",Loyal


,cohort,customers,net_revenue_share_pct
0,Top 10 customers,10,16.51
1,Top 10% of customers,434,59.89
2,Top 20% of customers,868,73.66


identified-customer net revenue: £8,288,931
company net revenue            : £9,748,131 (difference = unidentified-customer revenue, expected)


## 17. Executive KPI Summary

One table, one grain: the whole business over the full period. This is the
table the Power BI KPI cards read.


In [20]:
def build_executive_kpis(facts, orders_df, customers_df) -> pd.DataFrame:
    so = orders_df[orders_df["is_sale_order"]]
    identified = customers_df

    gross = facts["gross_revenue"].sum()
    canc  = facts["cancellation_value"].sum()
    net   = facts["net_revenue"].sum()
    n_orders   = so["InvoiceNo"].nunique()
    n_cancel   = orders_df.loc[~orders_df["is_sale_order"], "InvoiceNo"].nunique()
    n_customers = identified["CustomerID"].nunique()
    n_units    = so["order_units"].sum()

    kpis = [
        ("Gross Revenue (GBP)",          gross,  "sum of signed revenue on SALE lines"),
        ("Cancellation Value (GBP)",     canc,   "abs(signed revenue) on CANCELLATION lines"),
        ("Net Revenue (GBP)",            net,    "gross revenue - cancellation value"),
        ("Orders",                       n_orders, "distinct valid-sale invoices"),
        ("Customers (identified)",       n_customers, "distinct CustomerID with >=1 sale"),
        ("Units Sold",                   n_units,  "quantity on valid-sale lines"),
        ("AOV (GBP)",                    gross / n_orders, "gross revenue / valid-sale orders"),
        ("Revenue per Customer (GBP)",   net / n_customers, "net revenue / identified customers"),
        ("Repeat Customer Rate (%)",     identified["is_repeat_customer"].mean() * 100,
         "identified customers with 2+ sale orders"),
        ("Cancellation Rate (%)",        n_cancel / (n_orders + n_cancel) * 100,
         "cancellation invoices / all invoices"),
    ]
    return pd.DataFrame(kpis, columns=["kpi", "value", "definition"])

executive_kpis = build_executive_kpis(fact, orders, customers)
display(executive_kpis)


,kpi,value,definition
0,Gross Revenue (GBP),"10,642,110.80",sum of signed revenue on SALE lines
1,Cancellation Value (GBP),"893,979.73",abs(signed revenue) on CANCELLATION lines
2,Net Revenue (GBP),"9,748,131.07",gross revenue - cancellation value
3,Orders,"19,960.00",distinct valid-sale invoices
4,Customers (identified),"4,338.00",distinct CustomerID with >=1 sale
5,Units Sold,"5,572,420.00",quantity on valid-sale lines
6,AOV (GBP),533.17,gross revenue / valid-sale orders
7,Revenue per Customer (GBP),"2,247.15",net revenue / identified customers
8,Repeat Customer Rate (%),65.58,identified customers with 2+ sale orders
9,Cancellation Rate (%),16.12,cancellation invoices / all invoices


## 18. Validation / Reconciliation (mandatory)

Automated tests proving the pipeline did not lose or invent value. Floating-point
comparisons use `np.isclose` (never `==`). A failure raises `AssertionError`
with a readable message; the full matrix is exported as `validation_report.csv`.


In [21]:
validation_rows = []

def check(name, expected, actual, kind="exact"):
    if kind == "close":
        ok = bool(np.isclose(expected, actual, rtol=1e-6, atol=1e-2))
    else:
        ok = expected == actual
    validation_rows.append({"test": name, "expected": expected,
                            "actual": actual, "status": "PASS" if ok else "FAIL"})
    return ok

# ---- row-level reconciliation -------------------------------------------------
check("raw row count == 541909 (source of truth)", 541909, len(raw))
check("clean rows = raw - exact duplicates",
      len(raw) - int(quality_report.loc[quality_report['check'] == 'duplicate_full_rows',
                                        'affected_rows'].iloc[0]), len(clean))
check("fact rows = sales + cancellations",
      int((fact['transaction_class'] == 'SALE').sum()
          + (fact['transaction_class'] == 'CANCELLATION').sum()), len(fact))
_excluded = clean[~clean["transaction_class"].isin(["SALE", "CANCELLATION"])]
check("fact rows + excluded (ADJUSTMENT/INVALID) == clean rows",
      len(clean), len(fact) + len(_excluded))

# ---- revenue reconciliation ----------------------------------------------------
gross = fact["gross_revenue"].sum(); canc_v = fact["cancellation_value"].sum()
net   = fact["net_revenue"].sum();  signed = fact["signed_revenue"].sum()
check("net == gross - cancellation", gross - canc_v, net, "close")
check("net == signed revenue (fact)", signed, net, "close")
check("orders.net_revenue sum == fact net", orders["order_revenue"].sum(), net, "close")
# customers table = identified customers WITH >=1 valid-sale order. A small set of
# customers exist ONLY on cancellation invoices (never a completed sale) - they are
# identified but carry no positive revenue, so they are documented, not dropped silently.
_cancel_only = set(fact.loc[fact["has_customer_id"] & ~fact["is_valid_sale"], "CustomerID"].dropna().unique()) \
             - set(customers["CustomerID"].unique())
_cancel_only_net = fact.loc[fact["CustomerID"].isin(_cancel_only), "net_revenue"].sum()
print(f"note: {len(_cancel_only)} cancel-only customers, net value £{_cancel_only_net:,.2f} "
      f"(no completed sale -> excluded from customer grain, documented here)")
check("customers net + unidentified net + cancel-only net == fact net",
      customers["net_revenue"].sum()
      + fact.loc[~fact["has_customer_id"], "net_revenue"].sum()
      + _cancel_only_net, net, "close")
check("monthly net sums == fact net", monthly_kpis["net_revenue"].sum(), net, "close")
check("country net sums == fact net", country_kpis["net_revenue"].sum(), net, "close")

# ---- grain reconciliation --------------------------------------------------------
check("unique invoices in fact == orders rows", fact["InvoiceNo"].nunique(), len(orders))
check("identified customers WITH >=1 sale == customers rows",
      fact.loc[fact["has_customer_id"] & fact["is_valid_sale"], "CustomerID"].nunique(),
      len(customers))
check("identified customers total == customers rows + cancel-only customers",
      fact.loc[fact["has_customer_id"], "CustomerID"].nunique(),
      len(customers) + len(_cancel_only))
check("AOV: gross / sale orders == executive AOV",
      executive_kpis.loc[executive_kpis["kpi"] == "AOV (GBP)", "value"].iloc[0],
      gross / orders.loc[orders["is_sale_order"], "InvoiceNo"].nunique(), "close")
check("cancellation value == abs(signed) on cancellations",
      fact.loc[~fact["is_valid_sale"], "signed_revenue"].abs().sum(), canc_v, "close")

# ---- time-dimension integrity ---------------------------------------------------
_day_net  = fact.groupby(fact["InvoiceDate"].dt.normalize())["net_revenue"].sum()
_mon_net  = fact.groupby(fact["InvoiceDate"].dt.strftime("%Y-%m"))["net_revenue"].sum()
check("daily facts roll up to monthly facts exactly", _day_net.sum(), _mon_net.sum(), "close")
check("monthly_kpis rows == months observed in fact", _mon_net.size, len(monthly_kpis))

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

failed = validation_report[validation_report["status"] == "FAIL"]
assert failed.empty, f"VALIDATION FAILED:\n{failed}"
print(f"\nAll {len(validation_report)} validation checks PASSED - pipeline reconciles to source.")


note: 33 cancel-only customers, net value £-10,411.24 (no completed sale -> excluded from customer grain, documented here)


,test,expected,actual,status
0,raw row count == 541909 (source of truth),"541,909.00","541,909.00",PASS
1,clean rows = raw - exact duplicates,"536,641.00","536,641.00",PASS
2,fact rows = sales + cancellations,"534,129.00","534,129.00",PASS
3,fact rows + excluded (ADJUSTMENT/INVALID) == c...,"536,641.00","536,641.00",PASS
4,net == gross - cancellation,"9,748,131.07","9,748,131.07",PASS
5,net == signed revenue (fact),"9,748,131.07","9,748,131.07",PASS
6,orders.net_revenue sum == fact net,"9,748,131.07","9,748,131.07",PASS
7,customers net + unidentified net + cancel-only...,"9,748,131.07","9,748,131.07",PASS
8,monthly net sums == fact net,"9,748,131.07","9,748,131.07",PASS
9,country net sums == fact net,"9,748,131.07","9,748,131.07",PASS



All 17 validation checks PASSED - pipeline reconciles to source.


## 19. Power BI Export Layer (CSV)

Twelve clean tables, snake_case headers, no Python objects, UTF-8. These are
the files Power BI connects to (Get Data → Text/CSV). Redundancy between
`country_kpis` / `dim_country` etc. is intentional: **dims are narrow
membership tables, KPI tables are pre-aggregated for one-click visuals**.


In [22]:
# ---- prep export-friendly frames ---------------------------------------------------
def snake(s):  # Power BI friendlier headers
    return (s.strip().lower().replace(" ", "_").replace("(", "").replace(")", "")
             .replace("%", "pct").replace("/", "_per_"))

fact_export = fact.rename(columns={
    "InvoiceNo": "invoice_no", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "invoice_date", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country"})
orders_export = orders.rename(columns={
    "InvoiceNo": "invoice_no", "InvoiceDate": "invoice_date", "CustomerID": "customer_id",
    "Country": "country"})

# ---- star-schema keys: unique, Power BI-joinable ---------------------------------
# order_key = invoice_no (unique in orders after order-grain aggregation)
# line_id   = surrogate row id (transaction lines have no natural unique key)
# date_key  = YYYYMMDD integer matching 06_dim_date for relationship creation
orders_export.insert(0, "order_key", orders_export["invoice_no"])
orders_export["date_key"] = orders_export["invoice_date"].dt.strftime("%Y%m%d").astype(int)
fact_export.insert(0, "line_id", np.arange(1, len(fact_export) + 1))
fact_export["order_key"] = fact_export["invoice_no"]
fact_export["date_key"] = fact_export["invoice_date"].dt.strftime("%Y%m%d").astype(int)
assert orders_export["order_key"].is_unique, "order_key must be unique for 1:* joins"

dim_customer = customers.copy()
dim_customer.columns = [snake(c) for c in dim_customer.columns]
dim_product = product_kpis[["StockCode", "Description"]].drop_duplicates().rename(
    columns={"StockCode": "stock_code", "Description": "description"})
dim_country = country_kpis[["Country"]].drop_duplicates().rename(columns={"Country": "country"})
dim_country["is_uk"] = dim_country["country"].eq("United Kingdom")

def export_csv(df, name):
    path = DIR_CSV / name
    df.to_csv(path, index=False, encoding="utf-8-sig")  # utf-8-sig -> clean Excel/Power BI read
    return path, len(df)

csv_exports = {
    "01_fact_transactions.csv":   fact_export,
    "02_fact_orders.csv":         orders_export,
    "03_dim_customer.csv":        dim_customer,
    "04_dim_product.csv":         dim_product,
    "05_dim_country.csv":         dim_country,
    "06_dim_date.csv":            dim_date,
    "07_monthly_kpis.csv":        monthly_kpis,
    "08_country_kpis.csv":        country_kpis,
    "09_product_kpis.csv":        product_kpis,
    "10_customer_kpis.csv":       dim_customer,          # same grain, KPI-rich copy
    "11_executive_kpis.csv":      executive_kpis,
    "12_data_quality_report.csv": quality_report,
}
print("CSV exports:")
for name, df in csv_exports.items():
    path, n = export_csv(df, name)
    print(f"  {name:<32} {n:>8,} rows  -> {path}")

# validation report joins the export layer for governance
export_csv(validation_report, "13_validation_report.csv")
export_csv(cleaning_report, "14_cleaning_log.csv")


CSV exports:
  01_fact_transactions.csv          534,129 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/01_fact_transactions.csv
  02_fact_orders.csv                 23,796 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/02_fact_orders.csv
  03_dim_customer.csv                 4,338 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/03_dim_customer.csv
  04_dim_product.csv                  4,148 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/04_dim_product.csv
  05_dim_country.csv                     38 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/05_dim_country.csv
  06_dim_date.csv                       374 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/06_dim_date.csv
  07_monthly_kpis.csv                    13 rows  -> /content/day-01-executive-sales-diagnostic/outputs/csv/07_monthly_kpis.csv
  08_country_kpis.csv                    38 rows  -> /content/day-01-executive-sales-diagnost

(PosixPath('/content/day-01-executive-sales-diagnostic/outputs/csv/14_cleaning_log.csv'),
 3)

## 20. Parquet Exports

**Why Parquet?** Columnar + typed + compressed → 5–10× smaller than CSV,
preserves dtypes (no re-parsing dates), and loads ~10× faster in pandas /
Power BI (via the Parquet connector) / Spark. CSV stays for maximum
compatibility; Parquet is the performance path.


In [23]:
parquet_exports = {
    "clean_transactions.parquet": fact_export,
    "orders.parquet":             orders_export,
    "customers.parquet":          dim_customer,
    "products.parquet":           product_kpis,
    "monthly_kpis.parquet":       monthly_kpis,
    "country_kpis.parquet":       country_kpis,
}
print("Parquet exports:")
for name, df in parquet_exports.items():
    path = DIR_PARQUET / name
    df.to_parquet(path, index=False)
    print(f"  {name:<32} {len(df):>8,} rows  {path.stat().st_size / 1024:>8.0f} KB -> {path}")

# NOTE: the zipped download bundle is created at the very END of the notebook
# (after recommendations are written) so every export is included.


Parquet exports:
  clean_transactions.parquet        534,129 rows      9383 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/clean_transactions.parquet
  orders.parquet                     23,796 rows       899 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/orders.parquet
  customers.parquet                   4,338 rows       265 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/customers.parquet
  products.parquet                    4,148 rows       241 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/products.parquet
  monthly_kpis.parquet                   13 rows        11 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/monthly_kpis.parquet
  country_kpis.parquet                   38 rows        12 KB -> /content/day-01-executive-sales-diagnostic/outputs/parquet/country_kpis.parquet


## 21. Plotly Analysis

Ten decision-oriented charts. Power BI remains the executive dashboard; these
are the analyst's working visuals. The final month (Dec 2011, 9 days) is shown
but annotated where it distorts a trend.


In [24]:
TREND = "#1f77b4"; WARN = "#d62728"; ACCENT = "#2ca02c"; NEUTRAL = "#7f7f7f"
mk = monthly_kpis.copy()

def save_fig(fig, name):
    out = PROJECT_DIR / "outputs" / "charts"
    out.mkdir(parents=True, exist_ok=True)
    fig.write_html(out / f"{name}.html", include_plotlyjs="cdn")

# 1 — Monthly net revenue trend -------------------------------------------------
fig = px.line(mk, x="year_month", y="net_revenue", markers=True,
              title="Monthly Net Revenue — growth with seasonal spike",
              labels={"year_month": "Month", "net_revenue": "Net Revenue (GBP)"},
              hover_data={"gross_revenue": ":,.0f", "cancellation_value": ":,.0f"})
fig.update_traces(line_color=TREND); save_fig(fig, "01_monthly_revenue"); fig.show()

# 2 — Monthly orders -------------------------------------------------------------
fig = px.line(mk, x="year_month", y="orders", markers=True,
              title="Monthly Valid-Sale Orders",
              labels={"year_month": "Month", "orders": "Orders"})
fig.update_traces(line_color=ACCENT); save_fig(fig, "02_monthly_orders"); fig.show()

# 3 — Monthly AOV -----------------------------------------------------------------
fig = px.line(mk, x="year_month", y="aov", markers=True,
              title="Monthly Average Order Value (AOV)",
              labels={"year_month": "Month", "aov": "AOV (GBP)"})
fig.update_traces(line_color=NEUTRAL); save_fig(fig, "03_monthly_aov"); fig.show()

# 4 — Repeat vs one-time customers ---------------------------------------------------
seg_counts = customers["customer_segment"].value_counts().reindex(["One-Time", "Returning", "Loyal"])
fig = px.bar(x=seg_counts.index, y=seg_counts.values, text=seg_counts.values,
             title="Customer Base by Segment (identified customers)",
             labels={"x": "Segment", "y": "Customers"}, color=seg_counts.index,
             color_discrete_map={"One-Time": WARN, "Returning": TREND, "Loyal": ACCENT})
fig.update_traces(texttemplate="%{text:,}", textposition="outside"); fig.update_layout(showlegend=False)
save_fig(fig, "04_customer_segments"); fig.show()

# 5 — Top 10 countries by net revenue -------------------------------------------------
top_c = country_kpis.head(10).sort_values("net_revenue")
fig = px.bar(top_c, x="net_revenue", y="Country", orientation="h",
             title="Top 10 Countries by Net Revenue",
             labels={"net_revenue": "Net Revenue (GBP)", "Country": ""},
             hover_data={"repeat_customer_rate": ":.1f", "cancellation_rate": ":.1f"})
fig.update_traces(marker_color=TREND); save_fig(fig, "05_top_countries"); fig.show()

# 6 — Top 10 products by net revenue -----------------------------------------------------
top_p = product_kpis.head(10).sort_values("net_revenue")
fig = px.bar(top_p, x="net_revenue", y="Description", orientation="h",
             title="Top 10 Products by Net Revenue",
             labels={"net_revenue": "Net Revenue (GBP)", "Description": ""},
             hover_data={"units": ":,", "orders": ":,"})
fig.update_traces(marker_color=ACCENT); save_fig(fig, "06_top_products"); fig.show()

# 7 — Cancellation rate by month ----------------------------------------------------------
fig = px.bar(mk, x="year_month", y="cancellation_rate",
             title="Cancellation Rate by Month",
             labels={"year_month": "Month", "cancellation_rate": "Cancellation Rate (%)"},
             hover_data={"cancel_orders": ":,", "cancellation_value": ":,.0f"})
fig.update_traces(marker_color=WARN); save_fig(fig, "07_cancellation_rate"); fig.show()

# 8 — Revenue distribution by customer segment ------------------------------------------------
seg_rev = customers.groupby("customer_segment")["net_revenue"].sum().reindex(["One-Time", "Returning", "Loyal"])
fig = px.pie(values=seg_rev.values, names=seg_rev.index, hole=0.45,
             title="Net Revenue Share by Customer Segment",
             color=seg_rev.index,
             color_discrete_map={"One-Time": WARN, "Returning": TREND, "Loyal": ACCENT})
fig.update_traces(textinfo="percent+label"); save_fig(fig, "08_segment_revenue"); fig.show()

# 9 — Customer revenue concentration (Pareto) ----------------------------------------------------
pareto = customers.sort_values("net_revenue", ascending=False).reset_index(drop=True)
pareto["cum_share"] = pareto["net_revenue"].cumsum() / pareto["net_revenue"].sum() * 100
pareto["customer_rank_pct"] = (pareto.index + 1) / len(pareto) * 100
fig = px.line(pareto, x="customer_rank_pct", y="cum_share",
              title="Customer Revenue Concentration (Pareto) — top 10% of customers drive most revenue",
              labels={"customer_rank_pct": "Customers (top %)", "cum_share": "Cumulative Net Revenue (%)"})
fig.add_hline(y=80, line_dash="dot", line_color=NEUTRAL)
fig.update_traces(line_color=TREND); save_fig(fig, "09_concentration"); fig.show()

# 10 — Revenue decomposition indicators ---------------------------------------------------------------
fig = go.Figure(go.Bar(
    x=["Active Customers", "Orders per Customer", "AOV", "Net Revenue"],
    y=[cust_growth, freq_growth, aov_growth, rev_growth],
    text=[f"{v:+.1f}%" for v in [cust_growth, freq_growth, aov_growth, rev_growth]],
    textposition="outside",
    marker_color=[TREND, ACCENT, NEUTRAL, WARN]))
fig.update_layout(title=f"Revenue Decomposition — {first_m['year_month']} to {last_m['year_month']} (full months)",
                  yaxis_title="Change (%)")
save_fig(fig, "10_decomposition"); fig.show()


## 22. Executive Findings (computed, not invented)

Every number below is derived live from the dataset in this run.


In [25]:
ek = executive_kpis.set_index("kpi")["value"]
top_country = country_kpis.iloc[0]
onetime_share = (customers["customer_segment"] == "One-Time").mean() * 100
loyal_rev_share = (customers.loc[customers["customer_segment"] == "Loyal", "net_revenue"].sum()
                   / customers["net_revenue"].sum() * 100)
unknown_rev_share = (fact.loc[~fact["has_customer_id"], "net_revenue"].sum()
                     / fact["net_revenue"].sum() * 100)
peak_month = monthly_kpis.loc[monthly_kpis["net_revenue"].idxmax()]
worst_cancel_month = monthly_kpis.loc[monthly_kpis["cancellation_rate"].idxmax()]

findings = f'''
EXECUTIVE FINDINGS — UCI Online Retail, {fact['InvoiceDate'].min():%b %Y} to {fact['InvoiceDate'].max():%b %Y}
{'=' * 78}
1. SCALE          Net revenue £{ek['Net Revenue (GBP)']:,.0f} on {ek['Orders']:,.0f} valid orders
                  from {ek['Customers (identified)']:,.0f} identified customers.
2. REVENUE QUALITY Cancellations destroyed £{ek['Cancellation Value (GBP)']:,.0f} of value —
                  {ek['Cancellation Value (GBP)'] / ek['Gross Revenue (GBP)'] * 100:.1f}% of gross sales never converted
                  to booked revenue (cancellation rate {ek['Cancellation Rate (%)']:.1f}% of invoices).
3. RETENTION GAP  Repeat customer rate is {ek['Repeat Customer Rate (%)']:.1f}% — {onetime_share:.1f}% of
                  identified customers bought exactly ONCE and never returned.
4. CONCENTRATION  Top 10% of customers generate {top10pct_share:.1f}% of identified net revenue;
                  top 20% generate {top20pct_share:.1f}%. High dependency on a narrow base.
5. GEOGRAPHY      {top_country['Country']} contributes £{top_country['net_revenue']:,.0f}
                  ({top_country['net_revenue'] / country_kpis['net_revenue'].sum() * 100:.1f}% of net revenue).
6. SEASONALITY    Peak month: {peak_month['year_month']} (£{peak_month['net_revenue']:,.0f}).
                  Worst cancellation month: {worst_cancel_month['year_month']}
                  ({worst_cancel_month['cancellation_rate']:.1f}% of invoices).
7. GROWTH DRIVER  Between {first_m['year_month']} and {last_m['year_month']} (full months), net revenue
                  moved {rev_growth:+.1f}%, most associated with active customers {cust_growth:+.1f}%,
                  orders/customer {freq_growth:+.1f}%, AOV {aov_growth:+.1f}% (association, not causation).
8. DATA CAVEAT    {unknown_rev_share:.1f}% of net revenue comes from orders with NO CustomerID —
                  invisible to every retention metric. Identity capture is a measurement risk.
'''
print(findings)

with open(DIR_REPORTS / "executive_findings.txt", "w") as f:
    f.write(findings)



EXECUTIVE FINDINGS — UCI Online Retail, Dec 2010 to Dec 2011
1. SCALE          Net revenue £9,748,131 on 19,960 valid orders
                  from 4,338 identified customers.
2. REVENUE QUALITY Cancellations destroyed £893,980 of value —
                  8.4% of gross sales never converted
                  to booked revenue (cancellation rate 16.1% of invoices).
3. RETENTION GAP  Repeat customer rate is 65.6% — 34.4% of
                  identified customers bought exactly ONCE and never returned.
4. CONCENTRATION  Top 10% of customers generate 59.9% of identified net revenue;
                  top 20% generate 73.7%. High dependency on a narrow base.
5. GEOGRAPHY      United Kingdom contributes £8,189,252
                  (84.0% of net revenue).
6. SEASONALITY    Peak month: 2011-11 (£1,456,146).
                  Worst cancellation month: 2011-01
                  (19.3% of invoices).
7. GROWTH DRIVER  Between 2011-01 and 2011-11 (full months), net revenue
                  move

## 23. Five Evidence-Backed Recommendations

Each recommendation is generated from the computed metrics above — the numbers
are inserted programmatically, so they always reflect the actual data.


In [26]:
recs = pd.DataFrame([
    {
        "finding": f"Repeat customer rate is only {ek['Repeat Customer Rate (%)']:.1f}% — {onetime_share:.1f}% of identified customers never bought twice.",
        "evidence": f"{int((~customers['is_repeat_customer']).sum()):,} of {len(customers):,} identified customers have exactly 1 order.",
        "implication": "Revenue growth is acquisition-driven and fragile; the repeat engine is underdeveloped.",
        "action": "Launch a post-first-purchase journey: 14/30/60-day win-back emails, second-order incentive, and a 'customers also bought' cross-sell flow.",
        "monitor_kpi": "Repeat Customer Rate; Orders per Customer; 90-day repurchase rate",
    },
    {
        "finding": f"Cancellations destroyed {ek['Cancellation Value (GBP)'] / ek['Gross Revenue (GBP)'] * 100:.1f}% of gross sales (£{ek['Cancellation Value (GBP)']:,.0f}).",
        "evidence": f"{orders.loc[~orders['is_sale_order'], 'InvoiceNo'].nunique():,} cancellation invoices; worst month {worst_cancel_month['year_month']} at {worst_cancel_month['cancellation_rate']:.1f}%.",
        "implication": "Reported revenue overstates booked revenue; ops issues (stock-outs, fulfilment, pricing errors) are leaking value.",
        "action": "Root-cause the top cancellation drivers by product/country; add pre-dispatch stock validation and an approval step for large manual cancellations.",
        "monitor_kpi": "Cancellation Rate; Cancellation Value; cancellation value by product",
    },
    {
        "finding": f"Top 10% of customers generate {top10pct_share:.1f}% of identified net revenue (top 20%: {top20pct_share:.1f}%).",
        "evidence": f"Loyal segment (4+ orders) alone accounts for {loyal_rev_share:.1f}% of identified net revenue.",
        "implication": "Losing a handful of top accounts moves the P&L; churn in the head of the curve is the biggest single risk.",
        "action": "Stand up a key-account program: named owners for the top 100 customers, quarterly reviews, priority fulfilment and early access.",
        "monitor_kpi": "Top-10% revenue share; Loyal-segment retention; key-account churn",
    },
    {
        "finding": f"{top_country['Country']} contributes {top_country['net_revenue'] / country_kpis['net_revenue'].sum() * 100:.1f}% of net revenue; several export markets show higher repeat rates at small scale.",
        "evidence": f"{country_kpis[country_kpis['identified_customers'] >= 50].nlargest(1, 'repeat_customer_rate').iloc[0]['Country']} leads repeat rate among meaningful markets.",
        "implication": "The domestic market funds the business, but replicable pockets of loyalty exist abroad.",
        "action": "Pick 2–3 high-repeat export markets for a localized retention pilot (shipping promise, local payment methods, targeted CRM).",
        "monitor_kpi": "Repeat rate by country; export-market revenue share",
    },
    {
        "finding": f"{unknown_rev_share:.1f}% of net revenue is attached to orders with no CustomerID.",
        "evidence": f"{int((~fact['has_customer_id']).sum()):,} fact rows are anonymous; they are excluded from every retention metric.",
        "implication": "Retention KPIs are systematically understated/distorted; you cannot manage what you cannot see.",
        "action": "Incentivize account creation at checkout (guest-checkout capture, loyalty signup) and backfill identity where possible.",
        "monitor_kpi": "% revenue with known CustomerID; identified-customer count",
    },
])
pd.set_option("display.max_colwidth", 120)
display(recs)
recs.to_csv(DIR_CSV / "15_recommendations.csv", index=False, encoding="utf-8-sig")

# ---- final zipped bundle (runs last so ALL exports are included) -------------------
zip_path = PROJECT_DIR / "outputs" / "day01_powerbi_exports.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(DIR_CSV.glob("*.csv")):
        zf.write(p, arcname=f"csv/{p.name}")
    for p in sorted(DIR_PARQUET.glob("*.parquet")):
        zf.write(p, arcname=f"parquet/{p.name}")
    zf.write(DIR_REPORTS / "executive_findings.txt", arcname="reports/executive_findings.txt")
print(f"\nbundle: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MB)")

if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))


,finding,evidence,implication,action,monitor_kpi
0,Repeat customer rate is only 65.6% — 34.4% of identified customers never bought twice.,"1,493 of 4,338 identified customers have exactly 1 order.",Revenue growth is acquisition-driven and fragile; the repeat engine is underdeveloped.,"Launch a post-first-purchase journey: 14/30/60-day win-back emails, second-order incentive, and a 'customers also bo...",Repeat Customer Rate; Orders per Customer; 90-day repurchase rate
1,"Cancellations destroyed 8.4% of gross sales (£893,980).","3,836 cancellation invoices; worst month 2011-01 at 19.3%.","Reported revenue overstates booked revenue; ops issues (stock-outs, fulfilment, pricing errors) are leaking value.",Root-cause the top cancellation drivers by product/country; add pre-dispatch stock validation and an approval step f...,Cancellation Rate; Cancellation Value; cancellation value by product
2,Top 10% of customers generate 59.9% of identified net revenue (top 20%: 73.7%).,Loyal segment (4+ orders) alone accounts for 80.2% of identified net revenue.,Losing a handful of top accounts moves the P&L; churn in the head of the curve is the biggest single risk.,"Stand up a key-account program: named owners for the top 100 customers, quarterly reviews, priority fulfilment and e...",Top-10% revenue share; Loyal-segment retention; key-account churn
3,United Kingdom contributes 84.0% of net revenue; several export markets show higher repeat rates at small scale.,Germany leads repeat rate among meaningful markets.,"The domestic market funds the business, but replicable pockets of loyalty exist abroad.","Pick 2–3 high-repeat export markets for a localized retention pilot (shipping promise, local payment methods, target...",Repeat rate by country; export-market revenue share
4,15.1% of net revenue is attached to orders with no CustomerID.,"132,565 fact rows are anonymous; they are excluded from every retention metric.",Retention KPIs are systematically understated/distorted; you cannot manage what you cannot see.,"Incentivize account creation at checkout (guest-checkout capture, loyalty signup) and backfill identity where possible.",% revenue with known CustomerID; identified-customer count



bundle: /content/day-01-executive-sales-diagnostic/outputs/day01_powerbi_exports.zip (20.7 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>